# NB9 — RF-DETR + FashionCLIP Core-7 Detection Smoke Test

Notebook này được viết lại từ đầu để kiểm tra **end-to-end** phần detection trên Google Colab.

Pipeline cần được xác nhận:

```text
input image
  -> RF-DETR Fashionpedia garment boxes
  -> garment crops
  -> frozen FashionCLIP image embedding (512-d, L2)
  -> cosine similarity với 7 Core-7 text prototypes
  -> coarse_category
  -> scorer handoff tensors
```

NB9 không suy diễn `master_category` cho ảnh user. Detector label chỉ dùng để lọc garment object hợp lệ; `coarse_category` đến từ FashionCLIP zero-shot Core-7.

**Điều kiện PASS:** cell cuối phải in chính xác `NB9 SMOKE PASS`.

## 1. Clean checkout

Notebook luôn làm việc với branch `feat/detection-rfdetr-fashionclip-core7`. Mỗi lần chạy, repo local trong `/content` được clone lại để tránh code/cache cũ từ `main` hoặc từ một lần chạy trước.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/detection-rfdetr-fashionclip-core7"
REPO_DIR = Path("/content/opisoverated")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Install runtime dependencies

Cell này **không dùng `--upgrade`**. Mục tiêu là giữ nguyên NumPy/SciPy có sẵn của Colab nếu chúng đã thỏa contract, tránh thay binary package ngay dưới một kernel đang chạy.

Nếu pip bắt buộc thay NumPy hoặc SciPy trong khi package đó đã được import, notebook sẽ dừng và yêu cầu restart runtime. Sau restart, chạy lại từ đầu.

In [ ]:
from importlib import metadata as importlib_metadata
from packaging.version import Version

REQUIREMENTS_PATH = REPO_DIR / "requirements-detection.txt"
print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

binary_packages = ("numpy", "scipy")
loaded_before = {
    name: getattr(sys.modules.get(name), "__version__", None)
    for name in binary_packages
    if name in sys.modules
}
dist_before = {}
for name in binary_packages:
    try:
        dist_before[name] = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        dist_before[name] = None

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_PATH)],
    check=True,
)

dist_after = {
    name: importlib_metadata.version(name)
    for name in binary_packages
}

changed_loaded = {
    name: (loaded_before[name], dist_after[name])
    for name in loaded_before
    if loaded_before[name] != dist_after[name]
}
if changed_loaded:
    raise RuntimeError(
        "A binary dependency changed on disk while its old version is already "
        f"loaded in this kernel: {changed_loaded}. Restart the Colab runtime, "
        "then Run all again."
    )

versions = {}
for name in ("torch", "numpy", "scipy", "transformers", "rfdetr", "huggingface_hub"):
    try:
        versions[name] = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        versions[name] = None

assert Version(versions["numpy"]) < Version("2.4")
assert Version(versions["transformers"]) >= Version("5.1")
assert Version(versions["transformers"]) < Version("6")

for name, version in versions.items():
    print(f"{name}=={version}")

## 3. Runtime compatibility preflight

Import toàn bộ dependency chain trước khi tải model. Cell này cố ý import `numpy.testing` để bắt sớm lỗi ABI/mixed NumPy kiểu `_blas_supports_fpe`.

In [ ]:
import numpy as np
import scipy
import torch
import transformers
import rfdetr
import numpy.testing._private.utils as numpy_testing_utils

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert np.isfinite(np.array([0.0, 1.0])).all()
assert hasattr(numpy_testing_utils, "assert_allclose")
print("runtime preflight: PASS")

## 4. Lightweight repository tests

Chạy contract/unit tests trước khi tải checkpoint nặng. Nếu phần taxonomy, crop logic, scorer handoff hoặc notebook contract hỏng thì dừng ở đây.

In [ ]:
test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_detection_core7.py",
        "-v",
    ],
    cwd=REPO_DIR,
    text=True,
)
if test_run.returncode != 0:
    raise RuntimeError("Detection contract tests failed.")
print("unit tests: PASS")

## 5. Load detection config và smoke image

NB9 dùng ảnh `tests/animage.jpg` đã commit cùng branch. Missing image là lỗi cứng, không được silently skip.

In [ ]:
import sys
from PIL import Image
from IPython.display import display

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.detection import CORE7_CATEGORIES, CORE7_CATEGORY_TO_ID, load_detection_config

CONFIG_PATH = REPO_DIR / "configs/detection_rfdetr_fashionclip_core7_v1.json"
IMAGE_PATH = REPO_DIR / "tests/animage.jpg"
OUTPUT_DIR = REPO_DIR / "outputs/nb9_detection_smoke"

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(IMAGE_PATH)

config = load_detection_config(CONFIG_PATH)
smoke_image = Image.open(IMAGE_PATH).convert("RGB")

print("config:", CONFIG_PATH)
print("image:", IMAGE_PATH)
print("image size:", smoke_image.size)
print("Core-7:", CORE7_CATEGORY_TO_ID)
display(smoke_image)

## 6. Run RF-DETR → FashionCLIP → Core-7

Đây là cell end-to-end chính. Lần chạy đầu có thể mất thời gian vì tải checkpoint RF-DETR và FashionCLIP.

In [ ]:
from src.detection import DetectionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = DetectionPipeline(config, device=device)

result, image = pipeline.run(IMAGE_PATH)

print("accepted garments:", len(result.garments))
print("rejected detections:", len(result.rejected_detections))
print("RF-DETR runtime ms:", result.detector_runtime_ms)

for index, garment in enumerate(result.garments):
    print(
        f"[{index}]",
        garment.candidate.detector_label,
        "->",
        garment.category.coarse_category,
        f"det_conf={garment.candidate.detector_confidence}",
        f"sim={garment.category.similarity:.4f}",
        f"margin={garment.category.margin:.4f}",
    )

if not result.garments:
    raise RuntimeError("RF-DETR/FashionCLIP produced zero accepted garments.")

## 7. Validate embeddings và Core-7 predictions

Mỗi accepted garment phải có embedding 512-d hữu hạn, gần unit norm, và một `coarse_category` canonical.

In [ ]:
embedding_checks = []
for index, garment in enumerate(result.garments):
    embedding = torch.as_tensor(garment.embedding, dtype=torch.float32)
    norm = float(torch.linalg.vector_norm(embedding))
    category = garment.category.coarse_category
    category_id = garment.category.coarse_category_id

    assert embedding.shape == (512,), (index, embedding.shape)
    assert bool(torch.isfinite(embedding).all()), index
    assert abs(norm - 1.0) < 1e-3, (index, norm)
    assert category in CORE7_CATEGORY_TO_ID, category
    assert category_id == CORE7_CATEGORY_TO_ID[category], (category, category_id)

    embedding_checks.append((index, category, category_id, norm))

for row in embedding_checks:
    print(row)
print("embedding/category contract: PASS")

## 8. Visualize accepted crops

Mỗi crop hiển thị detector label, Core-7 category, cosine similarity và margin.

In [ ]:
import matplotlib.pyplot as plt

for index, garment in enumerate(result.garments):
    crop = image.crop(garment.crop_box_xyxy)
    plt.figure(figsize=(4, 4))
    plt.imshow(crop)
    plt.title(
        f"{garment.candidate.detector_label}\n"
        f"{garment.category.coarse_category} | "
        f"sim={garment.category.similarity:.3f} | "
        f"margin={garment.category.margin:.3f}"
    )
    plt.axis("off")
    plt.show()

## 9. Save detection output + scorer handoff

`save_detection_result` phải tạo metadata, crops và — nếu số garment nằm trong scorer contract — `scorer_inputs.pt`.

In [ ]:
import json
from src.detection.pipeline import save_detection_result

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

saved = save_detection_result(
    result,
    image,
    OUTPUT_DIR,
    scorer_min_items=config.scorer_min_items,
    scorer_max_items=config.scorer_max_items,
)

print(json.dumps(saved, indent=2))
if saved["scorer_handoff_error"]:
    raise RuntimeError(saved["scorer_handoff_error"])

## 10. Validate scorer handoff

Scorer input contract cho một outfit:

```text
item_embeddings      [1, N, 512]
coarse_category_ids  [1, N]
item_mask            [1, N]
```

NB9 cũng xác nhận inference metadata không giả lập `master_category`.

In [ ]:
metadata_path = Path(saved["metadata_path"])
scorer_path = Path(saved["scorer_inputs_path"])

assert metadata_path.is_file()
assert scorer_path.is_file()

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
scorer_batch = torch.load(scorer_path, map_location="cpu")

n = len(result.garments)
assert scorer_batch["item_embeddings"].shape == (1, n, 512)
assert scorer_batch["coarse_category_ids"].shape == (1, n)
assert scorer_batch["item_mask"].shape == (1, n)
assert scorer_batch["item_mask"].dtype == torch.bool
assert bool(scorer_batch["item_mask"].all())

assert metadata["taxonomy"]["master_category"] is None
for garment_metadata in metadata["garments"]:
    assert "master_category" not in garment_metadata

print("item_embeddings:", tuple(scorer_batch["item_embeddings"].shape))
print("coarse_category_ids:", scorer_batch["coarse_category_ids"].tolist())
print("item_mask:", scorer_batch["item_mask"].tolist())
print("scorer handoff: PASS")

## 11. Final PASS gate

Chỉ cell này quyết định NB9 có pass smoke test end-to-end hay không.

In [ ]:
assert len(result.garments) >= config.scorer_min_items
assert len(result.garments) <= config.scorer_max_items
assert metadata_path.is_file()
assert scorer_path.is_file()
assert len(saved["crop_paths"]) == len(result.garments)
assert all(Path(path).is_file() for path in saved["crop_paths"])

print("=" * 60)
print("NB9 SMOKE PASS")
print(f"accepted garments: {len(result.garments)}")
print(f"output: {OUTPUT_DIR}")
print("=" * 60)